**Celda 1 — Conectar Google Drive**

In [1]:
# Conectamos al Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Celda 2 — Instalar y cargar librerías**

In [2]:
# Instalamos geopandas
!pip install geopandas --quiet

# Cargamos todas las librerías que usaremos en el notebook
import pandas as pd
import geopandas as gpd
import os

**Celda 3 — Definir la ruta base a tus datos**

In [3]:
# Ruta base — ajusta si tu carpeta tiene otro nombre
BASE = '/content/drive/MyDrive/TFM_Seguridad_Vial/datos'

# Verificamos que la ruta existe y vemos las carpetas
print(os.listdir(BASE))

['accidentes_bici', 'accidentes_general', 'emt_paradas', 'poblacion_distrito', 'infraestructura_ciclista', 'distritos']


# Bloque 1 — Accidentes ciclistas

**Celda 4 — Cargar y unir los 6 CSV de accidentes de bici**

In [4]:
# Cargamos los 6 ficheros de bici y los unimos en uno solo
carpeta_bici = os.path.join(BASE, 'accidentes_bici/raw')

ficheros_bici = [
    'AccidentesBici2019.csv',
    'AccidentesBici2020.csv',
    'AccidentesBici2021.csv',
    'AccidentesBici2022.csv',
    'AccidentesBici2023.csv',
    'AccidentesBici2024.csv',
    'AccidentesBici2025.csv'
]

lista_dfs = []
for fichero in ficheros_bici:
    ruta = os.path.join(carpeta_bici, fichero)
    df_temp = pd.read_csv(ruta, sep=';', encoding='utf-8-sig')
    lista_dfs.append(df_temp)
    print(f'{fichero}: {len(df_temp)} filas')

# Unimos todos en un único dataframe
bici = pd.concat(lista_dfs, ignore_index=True)
print(f'\nTOTAL filas: {len(bici)}')
print(f'Columnas: {bici.columns.tolist()}')

AccidentesBici2019.csv: 891 filas
AccidentesBici2020.csv: 929 filas
AccidentesBici2021.csv: 845 filas
AccidentesBici2022.csv: 878 filas
AccidentesBici2023.csv: 905 filas
AccidentesBici2024.csv: 1076 filas
AccidentesBici2025.csv: 1317 filas

TOTAL filas: 6841
Columnas: ['num_expediente', 'fecha', 'hora', 'localizacion', 'numero', 'cod_distrito', 'distrito', 'tipo_accidente', 'estado_meteorológico', 'tipo_vehículo', 'tipo_persona', 'rango_edad', 'sexo', 'cod_lesividad', 'tipo_lesividad', 'coordenada_x_utm', 'coordenada_y_utm', 'positiva_alcohol', 'positiva_droga', 'tipo_vehiculo', 'lesividad']


**Celda 5 — Limpiar y filtrar ciclistas**

In [5]:
# Unificamos nombres de columnas con tilde/sin tilde
bici = bici.rename(columns={
    'tipo_vehículo': 'tipo_vehiculo',
    'tipo_lesividad': 'lesividad'
})

# Eliminamos columnas duplicadas que puedan haber quedado
bici = bici.loc[:, ~bici.columns.duplicated()]

# Extraemos el año de la columna fecha
bici['año'] = pd.to_datetime(bici['fecha'], dayfirst=True).dt.year

# Verificamos tipos de persona que hay
print("Tipos de persona:", bici['tipo_persona'].unique())
print("Tipos de vehículo:", bici['tipo_vehiculo'].unique())

Tipos de persona: ['Conductor' 'Peatón' 'Pasajero']
Tipos de vehículo: ['Bicicleta' 'Bicicleta EPAC (pedaleo asistido)' nan]


In [6]:
# Como aparecia nan en tipos de vehiculo, he mirado la distribucion de nan,
# aunque no afecta al análisis porque estamos contando por num_expediente único, no por tipo de vehículo.
# Estos conductores sin vehículo asignado siguen siendo ciclistas válidos dentro del expediente.
print(bici['tipo_vehiculo'].isna().sum())
print(bici[bici['tipo_vehiculo'].isna()]['tipo_persona'].value_counts())

4176
tipo_persona
Conductor    3895
Peatón        255
Pasajero       26
Name: count, dtype: int64


**Celda 6 — Crear variable de gravedad**

In [7]:
# Mapeamos cod_lesividad a categorías de gravedad
mapa_gravedad = {
    1: 'leve', 2: 'leve', 5: 'leve', 6: 'leve', 7: 'leve',
    3: 'grave',
    4: 'fallecido',
    14: 'sin_asistencia', 77: 'desconocido'
}

bici['gravedad'] = bici['cod_lesividad'].map(mapa_gravedad).fillna('sin_asistencia')

# Comprobamos el resultado
print(bici['gravedad'].value_counts())

gravedad
leve              4865
sin_asistencia    1738
grave              228
fallecido           10
Name: count, dtype: int64


**Celda 7 corregida — Agregar ciclistas con tres niveles de gravedad**

In [8]:
# Celda 7 — Función de agregación con gravedad máxima por expediente

# Jerarquía de gravedad: fallecido > grave > leve > sin_asistencia > desconocido
jerarquia_gravedad = {
    'fallecido':      4,
    'grave':          3,
    'leve':           2,
    'sin_asistencia': 1,
    'desconocido':    0
}

jerarquia_inversa = {v: k for k, v in jerarquia_gravedad.items()}

def agregar_accidentes(df, grupo):
    """
    Agrega accidentes por distrito y año separando leves, graves y mortales.
    Cada expediente se clasifica por su víctima de MAYOR gravedad.
    Esto garantiza que leves + graves + mortales = total, sin sobreestimación.
    Criterio coherente con la metodología oficial de la DGT.

    grupo: string para el prefijo de columnas ('bici' o 'peatones')
    """

    # Paso 1: asignamos valor numérico de gravedad a cada fila
    df = df.copy()
    df['gravedad_num'] = df['gravedad'].map(jerarquia_gravedad).fillna(0)

    # Paso 2: por cada expediente, distrito y año — nos quedamos con la gravedad máxima
    gravedad_max = (
        df.groupby(['distrito', 'año', 'num_expediente'])['gravedad_num']
        .max()
        .reset_index()
    )
    gravedad_max['gravedad_final'] = gravedad_max['gravedad_num'].map(jerarquia_inversa)

    # Paso 3: ahora cada expediente tiene una única gravedad — agregamos
    base = (
        gravedad_max.groupby(['distrito', 'año'])['num_expediente']
        .nunique()
        .reset_index()
        .rename(columns={'num_expediente': f'acc_{grupo}_total'})
    )

    leves = (
        gravedad_max[gravedad_max['gravedad_final'] == 'leve']
        .groupby(['distrito', 'año'])['num_expediente']
        .nunique()
        .reset_index()
        .rename(columns={'num_expediente': f'acc_{grupo}_leves'})
    )

    graves = (
        gravedad_max[gravedad_max['gravedad_final'] == 'grave']
        .groupby(['distrito', 'año'])['num_expediente']
        .nunique()
        .reset_index()
        .rename(columns={'num_expediente': f'acc_{grupo}_graves'})
    )

    mortales = (
        gravedad_max[gravedad_max['gravedad_final'] == 'fallecido']
        .groupby(['distrito', 'año'])['num_expediente']
        .nunique()
        .reset_index()
        .rename(columns={'num_expediente': f'acc_{grupo}_mortales'})
    )

    # Paso 4: unimos todo
    result = base.merge(leves,   on=['distrito', 'año'], how='left')
    result = result.merge(graves,   on=['distrito', 'año'], how='left')
    result = result.merge(mortales, on=['distrito', 'año'], how='left')

    cols = [f'acc_{grupo}_leves', f'acc_{grupo}_graves', f'acc_{grupo}_mortales']
    result[cols] = result[cols].fillna(0).astype(int)

    # Paso 5: verificación — ahora sí debe cumplirse leves+graves+mortales <= total
    # (<= y no == porque sin_asistencia y desconocido no entran en ninguna categoría)
    result['_verif'] = (
        result[f'acc_{grupo}_leves'] +
        result[f'acc_{grupo}_graves'] +
        result[f'acc_{grupo}_mortales']
    )
    discrepancias = result[result['_verif'] > result[f'acc_{grupo}_total']]

    if len(discrepancias) > 0:
        print(f"⚠️ Discrepancias en {grupo}: {len(discrepancias)} filas")
        print(discrepancias)
    else:
        print(f"✅ {grupo}: verificación OK — leves+graves+mortales ≤ total en todas las filas")

    result = result.drop(columns=['_verif'])

    # Paso 6: resumen por año
    print(f"\nAccidentes {grupo} por año:")
    print(
        result.groupby('año')[
            [f'acc_{grupo}_total', f'acc_{grupo}_leves',
             f'acc_{grupo}_graves', f'acc_{grupo}_mortales']
        ].sum().to_string()
    )

    return result

# Ejecutamos para bici
accidentes_bici = agregar_accidentes(bici, 'bici')
print(f'\nForma: {accidentes_bici.shape}')
print(f'Columnas: {accidentes_bici.columns.tolist()}')

✅ bici: verificación OK — leves+graves+mortales ≤ total en todas las filas

Accidentes bici por año:
      acc_bici_total  acc_bici_leves  acc_bici_graves  acc_bici_mortales
año                                                                     
2019             785             632               24                  1
2020             829             656               41                  1
2021             765             578               36                  0
2022             777             591               27                  2
2023             810             595               30                  1
2024             972             716               36                  3
2025            1191             916               34                  2

Forma: (147, 6)
Columnas: ['distrito', 'año', 'acc_bici_total', 'acc_bici_leves', 'acc_bici_graves', 'acc_bici_mortales']


# Bloque 2 — Peatones

**Celda 8 — Cargar los 6 CSV de accidentes generales**

In [9]:
carpeta_general = os.path.join(BASE, 'accidentes_general/raw')

ficheros_general = [
    'Accidentes2019.csv',
    'Accidentes2020.csv',
    'Accidentes2021.csv',
    'Accidentes2022.csv',
    'Accidentes2023.csv',
    'Accidentes2024.csv',
    'Accidentes2025.csv'
]

lista_dfs = []
for fichero in ficheros_general:
    ruta = os.path.join(carpeta_general, fichero)
    df_temp = pd.read_csv(ruta, sep=';', encoding='utf-8-sig')
    lista_dfs.append(df_temp)
    print(f'{fichero}: {len(df_temp)} filas')

general = pd.concat(lista_dfs, ignore_index=True)
print(f'\nTOTAL filas: {len(general)}')

Accidentes2019.csv: 51811 filas
Accidentes2020.csv: 32433 filas
Accidentes2021.csv: 41783 filas
Accidentes2022.csv: 47053 filas
Accidentes2023.csv: 48830 filas
Accidentes2024.csv: 49340 filas
Accidentes2025.csv: 51067 filas

TOTAL filas: 322317


**Celda 9 — Limpiar y filtrar peatones**

In [10]:
# Unificamos nombres de columnas igual que con bici
general = general.rename(columns={
    'tipo_vehículo': 'tipo_vehiculo',
    'tipo_lesividad': 'lesividad'
})
general = general.loc[:, ~general.columns.duplicated()]

# Extraemos el año
general['año'] = pd.to_datetime(general['fecha'], dayfirst=True).dt.year

# Filtramos solo peatones
peatones = general[general['tipo_persona'] == 'Peatón'].copy()
print(f'Filas peatones: {len(peatones)}')
print(f'Años: {sorted(peatones["año"].unique())}')

Filas peatones: 10989
Años: [np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]


**Celda 10 corregida — Agregar peatones con tres niveles**

In [11]:
# Celda 10 — Gravedad y agregación peatones

# Misma variable de gravedad que usamos para bici
peatones['gravedad'] = peatones['cod_lesividad'].map(mapa_gravedad).fillna('sin_asistencia')
# Verificamos distribución
print("Distribución gravedad peatones:")
print(peatones['gravedad'].value_counts())
print(f"\nTotal filas peatones: {len(peatones)}")

# Usamos la misma función definida en celda 7
accidentes_peatones = agregar_accidentes(peatones, 'peatones')

# Verificamos resultado
print(f"\nForma tabla accidentes peatones: {accidentes_peatones.shape}")
print(f"Columnas: {accidentes_peatones.columns.tolist()}")
print(f"\nEjemplo (primeras 10 filas):")
print(accidentes_peatones.head(10).to_string())

# Comprobación adicional: totales por año
print("\nAccidentes peatones por año:")
print(
    accidentes_peatones.groupby('año')[
        ['acc_peatones_total', 'acc_peatones_leves',
         'acc_peatones_graves', 'acc_peatones_mortales']
    ].sum().to_string()
)

Distribución gravedad peatones:
gravedad
leve              7879
sin_asistencia    1895
grave             1125
fallecido           89
desconocido          1
Name: count, dtype: int64

Total filas peatones: 10989
✅ peatones: verificación OK — leves+graves+mortales ≤ total en todas las filas

Accidentes peatones por año:
      acc_peatones_total  acc_peatones_leves  acc_peatones_graves  acc_peatones_mortales
año                                                                                     
2019                1679                1313                  183                     15
2020                1016                 757                  133                     14
2021                1234                 930                  147                     10
2022                1513                1154                  149                      8
2023                1603                1127                  177                     10
2024                1496                1039             

In [12]:
# Comprobación de posibles expedientes solapados entre siniestros con bicicleta y peatones

exp_bici = set(bici['num_expediente'].dropna().unique())
exp_peat = set(peatones['num_expediente'].dropna().unique())

solapados = exp_bici.intersection(exp_peat)

print(f"Nº de expedientes presentes tanto en bici como en peatones: {len(solapados)}")

# Ver algunos ejemplos si existen
list(solapados)[:10]

Nº de expedientes presentes tanto en bici como en peatones: 430


['2023S038962',
 '2019S027484',
 '2024S031457',
 '2025S035401',
 '2023S037073',
 '2025S121025',
 '2021S004370',
 '2019S022968',
 '2022S015392',
 '2021S012749']

In [13]:
# Crear siniestros VRU únicos: peatones + bicicleta sin doble conteo por expediente

# 1. Unimos ambos subconjuntos
vru = pd.concat([
    bici.assign(origen_vru='bici'),
    peatones.assign(origen_vru='peaton')
], ignore_index=True)

# 2. Convertimos la gravedad a escala ordinal
orden_gravedad = {
    'desconocido': 0,
    'sin_asistencia': 1,
    'leve': 2,
    'grave': 3,
    'fallecido': 4
}

vru['gravedad_rank'] = vru['gravedad'].map(orden_gravedad)

# 3. Nos quedamos con una única fila por expediente, distrito y año,
# usando la máxima gravedad registrada
vru_unicos = (
    vru.groupby(['distrito', 'año', 'num_expediente'], as_index=False)
       .agg(gravedad_rank=('gravedad_rank', 'max'))
)

# 4. Recuperamos la etiqueta de gravedad
rank_a_gravedad = {v: k for k, v in orden_gravedad.items()}
vru_unicos['gravedad_max'] = vru_unicos['gravedad_rank'].map(rank_a_gravedad)

# 5. Agregamos por distrito y año
accidentes_vru_unicos = (
    vru_unicos
    .groupby(['distrito', 'año'])
    .agg(
        acc_vru_total=('num_expediente', 'nunique'),
        acc_vru_leves=('gravedad_max', lambda x: (x == 'leve').sum()),
        acc_vru_graves=('gravedad_max', lambda x: (x == 'grave').sum()),
        acc_vru_mortales=('gravedad_max', lambda x: (x == 'fallecido').sum())
    )
    .reset_index()
)

# 6. Calculamos siniestralidad ponderada VRU sin doble conteo
accidentes_vru_unicos['acc_ponderado'] = (
    accidentes_vru_unicos['acc_vru_leves'] * 1 +
    accidentes_vru_unicos['acc_vru_graves'] * 7 +
    accidentes_vru_unicos['acc_vru_mortales'] * 16
)

accidentes_vru_unicos.head()

,distrito,año,acc_vru_total,acc_vru_leves,acc_vru_graves,acc_vru_mortales,acc_ponderado
0,ARGANZUELA,2019,138,101,15,1,222
1,ARGANZUELA,2020,95,74,6,1,132
2,ARGANZUELA,2021,114,88,14,0,186
3,ARGANZUELA,2022,102,72,10,0,142
4,ARGANZUELA,2023,113,84,10,0,154


In [14]:
# Diagnóstico: ¿cuántos expedientes se solapan realmente?
exp_bici = set(bici['num_expediente'].dropna().unique())
exp_peat = set(peatones['num_expediente'].dropna().unique())
solapados = exp_bici.intersection(exp_peat)
print(f"Expedientes solapados: {len(solapados)}")

# ¿Cuánto cambia acc_vru_total con y sin deduplicación?
sin_dedup = accidentes_bici[['distrito','año','acc_bici_total']].merge(
    accidentes_peatones[['distrito','año','acc_peatones_total']],
    on=['distrito','año']
)
sin_dedup['vru_suma'] = sin_dedup['acc_bici_total'] + sin_dedup['acc_peatones_total']

con_dedup = accidentes_vru_unicos[['distrito','año','acc_vru_total']]

comp = sin_dedup.merge(con_dedup, on=['distrito','año'])
comp['diferencia'] = comp['vru_suma'] - comp['acc_vru_total']

print(f"\nDiferencia total acumulada (doble conteo): {comp['diferencia'].sum()}")
print(f"Diferencia media por distrito-año: {comp['diferencia'].mean():.2f}")
print(f"\nTop 5 más afectados:")
print(comp.nlargest(5, 'diferencia')[['distrito','año','vru_suma','acc_vru_total','diferencia']].to_string())

Expedientes solapados: 430

Diferencia total acumulada (doble conteo): 430
Diferencia media por distrito-año: 2.93

Top 5 más afectados:
           distrito   año  vru_suma  acc_vru_total  diferencia
76  MONCLOA-ARAVACA  2025       182            167          15
26           CENTRO  2024       297            283          14
21           CENTRO  2019       289            276          13
24           CENTRO  2022       253            240          13
1        ARGANZUELA  2020       107             95          12


# Bloque 3 — Población

**Celda 11 — Cargar y limpiar población**

In [15]:
# Cargamos el fichero histórico 2019-2023
ruta_pob = os.path.join(BASE, 'poblacion_distrito/raw/poblacion_distrito_2017-2023.csv')

pob = pd.read_csv(ruta_pob, sep=';', encoding='utf-8-sig')

# Filtramos solo filas de distrito (cod_barrio == cod_distrito)
pob['cod_distrito'] = pob['cod_distrito'].astype(str).str.strip()
pob['cod_barrio'] = pob['cod_barrio'].astype(str).str.strip()
pob = pob[pob['cod_barrio'] == pob['cod_distrito']].copy()

# Extraemos el año y le restamos 1 (padrón a 1 enero XXXX = datos del año XXXX-1)
pob['año_padron'] = pob['fecha'].str.extract(r'(\d{4})').astype(int)
pob['año'] = pob['año_padron'] - 1

# Nos quedamos solo con lo que necesitamos
pob = pob[['año', 'cod_distrito', 'distrito', 'num_personas']].copy()
pob.columns = ['año', 'cod_distrito', 'distrito', 'poblacion']

# Limpiamos el nombre del distrito (viene con espacio delante)
pob['distrito'] = pob['distrito'].str.strip().str.upper()

print(pob.head(10))
print(f'\nAños disponibles: {sorted(pob["año"].unique())}')
print(f'Forma: {pob.shape}')

    año cod_distrito             distrito poblacion
0  2023            1               CENTRO   145.411
1  2023            2           ARGANZUELA   156.559
2  2023            3               RETIRO   119.757
3  2023            4            SALAMANCA   149.778
4  2023            5            CHAMARTÍN   148.111
5  2023            6               TETUÁN   166.211
6  2023            7             CHAMBERÍ   141.984
7  2023            8  FUENCARRAL-EL PARDO   253.898
8  2023            9      MONCLOA-ARAVACA   125.223
9  2023           10               LATINA   250.396

Años disponibles: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Forma: (154, 4)


**Celda 12 — Añadir el año 2024**

In [16]:
# Cargamos el fichero del 2024 que limpiaste a mano
ruta_pob24 = os.path.join(BASE, 'poblacion_distrito/raw/poblacion_distrito_2024.csv')

pob24 = pd.read_csv(ruta_pob24, encoding='utf-8-sig')
print("Columnas 2024:", pob24.columns.tolist())
print(pob24.head(3))

Columnas 2024: ['distrito', 'poblacion']
         distrito poblacion
0      01. Centro   146,349
1  02. Arganzuela   158,760
2      03. Retiro   120,447


**Celda 13 — Limpiar población 2024 y unir con histórico**

In [17]:
# Limpiamos el fichero 2024
pob24['distrito'] = pob24['distrito'].str.replace(r'^\d+\.\s*', '', regex=True)
pob24['distrito'] = pob24['distrito'].str.strip().str.upper()
pob24['poblacion'] = pd.to_numeric(
    pob24['poblacion'].astype(str).str.replace('.', '').str.replace(',', ''),
    errors='coerce'
)
pob24['año'] = 2024
pob24['cod_distrito'] = range(1, len(pob24) + 1)
pob24 = pob24[['año', 'cod_distrito', 'distrito', 'poblacion']]

# Limpiamos también los puntos del histórico 2019-2023
pob['poblacion'] = pd.to_numeric(
    pob['poblacion'].astype(str).str.replace('.', '').str.replace(',', ''),
    errors='coerce'
)

# Verificación: no deben quedar nulos
assert pob['poblacion'].isnull().sum() == 0,   "⚠️ Nulos en población histórica"
assert pob24['poblacion'].isnull().sum() == 0, "⚠️ Nulos en población 2024"
print("✅ Población limpia sin nulos")

# Filtramos histórico solo 2019-2023 y unimos con 2024
pob_final = pd.concat([
    pob[pob['año'].between(2019, 2023)],
    pob24
], ignore_index=True)

print(f'Años finales: {sorted(pob_final["año"].unique())}')
print(f'Forma final: {pob_final.shape}')
print('\nEjemplo poblacion:')
print(pob_final.head(5))
print('\nDistritos:')
print(sorted(pob_final['distrito'].unique()))

✅ Población limpia sin nulos
Años finales: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Forma final: (131, 4)

Ejemplo poblacion:
    año cod_distrito    distrito  poblacion
0  2023            1      CENTRO     145411
1  2023            2  ARGANZUELA     156559
2  2023            3      RETIRO     119757
3  2023            4   SALAMANCA     149778
4  2023            5   CHAMARTÍN     148111

Distritos:
['ARGANZUELA', 'BARAJAS', 'CARABANCHEL', 'CENTRO', 'CHAMARTÍN', 'CHAMBERÍ', 'CIUDAD LINEAL', 'FUENCARRAL-EL PARDO', 'HORTALEZA', 'LATINA', 'MONCLOA-ARAVACA', 'MORATALAZ', 'PUENTE DE VALLECAS', 'RETIRO', 'SALAMANCA', 'SAN BLAS-CANILLEJAS', 'TETUÁN', 'TODOS', 'USERA', 'VICÁLVARO', 'VILLA DE VALLECAS', 'VILLAVERDE']


**Celda 14 — Eliminar fila total Madrid y verificar**

In [18]:
# Eliminamos la fila "TODOS" que es el total de Madrid, no un distrito
pob_final = pob_final[pob_final['distrito'] != 'TODOS'].copy()

# Verificamos
print(f'Forma final: {pob_final.shape}')  # debe ser (126, 4)
print(f'Distritos únicos: {pob_final["distrito"].nunique()}')  # debe ser 21
print(f'Años: {sorted(pob_final["año"].unique())}')

Forma final: (126, 4)
Distritos únicos: 21
Años: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


# Bloque 4 — Paradas EMT

**Celda 15 — Cargar paradas EMT**

In [19]:
from shapely.geometry import Point

# Cargamos el fichero de paradas EMT
ruta_emt = os.path.join(BASE, 'emt_paradas/raw/stopsemt.csv')
emt = pd.read_csv(ruta_emt, skiprows=1)

# Deduplicamos: una fila por parada física única
paradas = emt.drop_duplicates(subset='parada')[['parada', 'posX', 'posY']].copy()
paradas = paradas.dropna(subset=['posX', 'posY'])
print(f'Paradas únicas: {len(paradas)}')

# Creamos GeoDataFrame con coordenadas UTM zona 30N
paradas_geo = gpd.GeoDataFrame(
    paradas,
    geometry=gpd.points_from_xy(paradas['posX'], paradas['posY']),
    crs='EPSG:25830'
)
print(f'CRS paradas: {paradas_geo.crs}')
print(paradas_geo.head(3))

Paradas únicas: 4931
CRS paradas: EPSG:25830
   parada      posX       posY                geometry
0    4514  439221.0  4476870.0  POINT (439221 4476870)
1    4022  439177.0  4476578.0  POINT (439177 4476578)
2    3687  439204.0  4476389.0  POINT (439204 4476389)


**Celda 16 — Cargar shapefile de barrios/distritos**

In [20]:
ruta_barrios = os.path.join(BASE, 'distritos/raw/BARRIOS.shp')
barrios = gpd.read_file(ruta_barrios)

print(f'CRS: {barrios.crs}')
print(f'Columnas: {barrios.columns.tolist()}')
print(f'Filas: {len(barrios)}')
print(barrios.head(3).to_string())

CRS: EPSG:25830
Columnas: ['CODDIS', 'NOMDIS', 'COD_BAR', 'NOMBRE', 'COD_DIS_TX', 'BARRIO_MAY', 'COD_DISBAR', 'NUM_BAR', 'BARRIO_MT', 'COD_DISB', 'Area', 'geometry']
Filas: 131
  CODDIS  NOMDIS COD_BAR       NOMBRE COD_DIS_TX   BARRIO_MAY COD_DISBAR NUM_BAR    BARRIO_MT COD_DISB          Area                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

**Celda 17 — Crear capa de distritos y asignar paradas EMT**

In [21]:
# Creamos una capa de distritos disolviendo los barrios
distritos = barrios.dissolve(by='CODDIS', aggfunc='first')[['NOMDIS', 'geometry']].reset_index()
distritos['NOMDIS'] = distritos['NOMDIS'].str.strip().str.upper()
print(f'Distritos: {len(distritos)}')
print(distritos[['CODDIS', 'NOMDIS']].to_string())

# Cruzamos paradas con distritos
paradas_distrito = gpd.sjoin(paradas_geo, distritos, how='left', predicate='within')

# Contamos paradas por distrito
paradas_por_distrito = (
    paradas_distrito.groupby('NOMDIS')['parada']
    .count()
    .reset_index()
    .rename(columns={'NOMDIS': 'distrito', 'parada': 'n_paradas_emt'})
)

print(paradas_por_distrito)
print(f'\nTotal distritos con paradas: {len(paradas_por_distrito)}')

Distritos: 21
   CODDIS                 NOMDIS
0       1                 CENTRO
1      10                 LATINA
2      11            CARABANCHEL
3      12                  USERA
4      13     PUENTE DE VALLECAS
5      14              MORATALAZ
6      15          CIUDAD LINEAL
7      16              HORTALEZA
8      17             VILLAVERDE
9      18      VILLA DE VALLECAS
10     19              VICÁLVARO
11      2             ARGANZUELA
12     20  SAN BLAS - CANILLEJAS
13     21                BARAJAS
14      3                 RETIRO
15      4              SALAMANCA
16      5              CHAMARTÍN
17      6                 TETUÁN
18      7               CHAMBERÍ
19      8  FUENCARRAL - EL PARDO
20      9      MONCLOA - ARAVACA
                 distrito  n_paradas_emt
0              ARGANZUELA            160
1                 BARAJAS            161
2             CARABANCHEL            273
3                  CENTRO            185
4               CHAMARTÍN            314
5             

**Celda 18 — Normalizar nombres de distrito en EMT**

In [22]:
# Unificamos el formato de guiones
paradas_por_distrito['distrito'] = (
    paradas_por_distrito['distrito']
    .str.replace(' - ', '-', regex=False)
)

print(paradas_por_distrito['distrito'].tolist())

['ARGANZUELA', 'BARAJAS', 'CARABANCHEL', 'CENTRO', 'CHAMARTÍN', 'CHAMBERÍ', 'CIUDAD LINEAL', 'FUENCARRAL-EL PARDO', 'HORTALEZA', 'LATINA', 'MONCLOA-ARAVACA', 'MORATALAZ', 'PUENTE DE VALLECAS', 'RETIRO', 'SALAMANCA', 'SAN BLAS-CANILLEJAS', 'TETUÁN', 'USERA', 'VICÁLVARO', 'VILLA DE VALLECAS', 'VILLAVERDE']


# Bloque 5 — Infraestructura ciclista

**Celda 19 — Calcular km de carril bici por distrito**

In [23]:
# Cargamos el shapefile de infraestructura ciclista
ruta_bici = os.path.join(BASE, 'infraestructura_ciclista/raw/Infraestructura_Ciclista.shp')
carriles = gpd.read_file(ruta_bici)

print(f'CRS carriles: {carriles.crs}')
print(f'Columnas: {carriles.columns.tolist()}')
print(f'Filas: {len(carriles)}')

CRS carriles: EPSG:25830
Columnas: ['COD_TIPOLO', 'COD_TIPO_V', 'Shape_STLe', 'd_COD_TIPO', 'd_COD_TI_1', 'geometry']
Filas: 1437


**Celda 20 — Calcular km de carril bici por distrito**

In [24]:
# Aseguramos mismo CRS
carriles = carriles.to_crs(distritos.crs)

# Intersección espacial: qué tramos caen en cada distrito
carriles_distrito = gpd.overlay(carriles, distritos, how='intersection')

# Calculamos longitud de cada tramo en metros
carriles_distrito['longitud_m'] = carriles_distrito.geometry.length

# Agregamos por distrito en km
km_carril_bici = (
    carriles_distrito.groupby('NOMDIS')['longitud_m']
    .sum()
    .div(1000)
    .reset_index()
    .rename(columns={'NOMDIS': 'distrito', 'longitud_m': 'km_carril_bici'})
)

# Normalizamos nombre igual que EMT
km_carril_bici['distrito'] = km_carril_bici['distrito'].str.replace(' - ', '-', regex=False)
km_carril_bici['km_carril_bici'] = km_carril_bici['km_carril_bici'].round(2)

print(km_carril_bici)
print(f'\nTotal distritos: {len(km_carril_bici)}')
print(f'Total km carriles bici en Madrid: {km_carril_bici["km_carril_bici"].sum().round(1)}')

               distrito  km_carril_bici
0            ARGANZUELA           30.32
1               BARAJAS           22.35
2           CARABANCHEL           30.51
3                CENTRO           40.78
4             CHAMARTÍN           43.54
5              CHAMBERÍ           30.86
6         CIUDAD LINEAL           22.93
7   FUENCARRAL-EL PARDO           61.53
8             HORTALEZA           48.65
9                LATINA           12.72
10      MONCLOA-ARAVACA          100.63
11            MORATALAZ           29.90
12   PUENTE DE VALLECAS           42.10
13               RETIRO           26.14
14            SALAMANCA           43.13
15  SAN BLAS-CANILLEJAS           38.81
16               TETUÁN           29.70
17                USERA           68.34
18            VICÁLVARO           43.69
19    VILLA DE VALLECAS           40.88
20           VILLAVERDE           26.31

Total distritos: 21
Total km carriles bici en Madrid: 833.8


# Bloque final — la tabla maestra

**Celda 21 corregida — Tabla maestra con lista fija de 21 distritos**

In [25]:
# Celda 21 — Construir la tabla maestra distrito × año

import itertools

distritos_lista = sorted(
    distritos['NOMDIS']
    .str.strip()
    .str.upper()
    .str.replace(' - ', '-', regex=False)
    .unique()
)
años_lista = [2019, 2020, 2021, 2022, 2023, 2024]

print(f'Distritos: {len(distritos_lista)}')
print(distritos_lista)
print(f'Años: {años_lista}')

if len(distritos_lista) != 21:
    print(f"⚠️ Atención: se esperaban 21 distritos, hay {len(distritos_lista)}")
else:
    print("✅ 21 distritos correctos")

tabla = pd.DataFrame(
    list(itertools.product(distritos_lista, años_lista)),
    columns=['distrito', 'año']
)
print(f'\nCombinaciones totales: {len(tabla)}')

# Unimos accidentes de bici y peatones por separado (para análisis individual)
tabla = tabla.merge(accidentes_bici,    on=['distrito', 'año'], how='left')
tabla = tabla.merge(accidentes_peatones, on=['distrito', 'año'], how='left')

cols_acc = [
    'acc_bici_total',     'acc_bici_leves',     'acc_bici_graves',     'acc_bici_mortales',
    'acc_peatones_total', 'acc_peatones_leves',  'acc_peatones_graves', 'acc_peatones_mortales'
]
tabla[cols_acc] = tabla[cols_acc].fillna(0).astype(int)

# Unimos población
tabla = tabla.merge(
    pob_final[['año', 'distrito', 'poblacion']],
    on=['distrito', 'año'], how='left'
)

# Unimos paradas EMT y km carril bici
tabla = tabla.merge(paradas_por_distrito, on='distrito', how='left')
tabla = tabla.merge(km_carril_bici,       on='distrito', how='left')

# ── VRU sin doble conteo — desde accidentes_vru_unicos ───────────────────
tabla = tabla.merge(
    accidentes_vru_unicos[[
        'distrito', 'año',
        'acc_vru_total', 'acc_vru_leves', 'acc_vru_graves',
        'acc_vru_mortales', 'acc_ponderado'
    ]],
    on=['distrito', 'año'], how='left'
)
cols_vru = ['acc_vru_total', 'acc_vru_leves', 'acc_vru_graves',
            'acc_vru_mortales', 'acc_ponderado']
tabla[cols_vru] = tabla[cols_vru].fillna(0)

# Verificación
assert (tabla['acc_vru_leves'] + tabla['acc_vru_graves'] + tabla['acc_vru_mortales']
        <= tabla['acc_vru_total']).all(), "⚠️ Inconsistencia en VRU"
print("✅ acc_vru sin doble conteo — expedientes solapados resueltos")

# Resumen
print(f'\nForma de la tabla: {tabla.shape}')
print(f'\nNulos por columna:')
print(tabla.isnull().sum())
print(f'\nEjemplo (primeras 5 filas):')
print(tabla.head(5).to_string())

conteo = tabla.groupby('distrito')['año'].count()
if (conteo != 6).any():
    print(f"\n⚠️ Algunos distritos no tienen 6 años:")
    print(conteo[conteo != 6])
else:
    print(f"\n✅ Todos los distritos tienen exactamente 6 años")

Distritos: 21
['ARGANZUELA', 'BARAJAS', 'CARABANCHEL', 'CENTRO', 'CHAMARTÍN', 'CHAMBERÍ', 'CIUDAD LINEAL', 'FUENCARRAL-EL PARDO', 'HORTALEZA', 'LATINA', 'MONCLOA-ARAVACA', 'MORATALAZ', 'PUENTE DE VALLECAS', 'RETIRO', 'SALAMANCA', 'SAN BLAS-CANILLEJAS', 'TETUÁN', 'USERA', 'VICÁLVARO', 'VILLA DE VALLECAS', 'VILLAVERDE']
Años: [2019, 2020, 2021, 2022, 2023, 2024]
✅ 21 distritos correctos

Combinaciones totales: 126
✅ acc_vru sin doble conteo — expedientes solapados resueltos

Forma de la tabla: (126, 18)

Nulos por columna:
distrito                 0
año                      0
acc_bici_total           0
acc_bici_leves           0
acc_bici_graves          0
acc_bici_mortales        0
acc_peatones_total       0
acc_peatones_leves       0
acc_peatones_graves      0
acc_peatones_mortales    0
poblacion                0
n_paradas_emt            0
km_carril_bici           0
acc_vru_total            0
acc_vru_leves            0
acc_vru_graves           0
acc_vru_mortales         0
acc_ponderado 

**Celda 22 — Guardar la tabla maestra**

In [26]:
# Guardamos la tabla maestra en outputs
ruta_output = '/content/drive/MyDrive/TFM_Seguridad_Vial/outputs/tabla_maestra.csv'

tabla.to_csv(ruta_output, index=False, encoding='utf-8-sig')

print('✅ Tabla maestra guardada correctamente')
print(f'Ruta: {ruta_output}')
print(f'Filas: {len(tabla)}')
print(f'Columnas: {tabla.columns.tolist()}')

✅ Tabla maestra guardada correctamente
Ruta: /content/drive/MyDrive/TFM_Seguridad_Vial/outputs/tabla_maestra.csv
Filas: 126
Columnas: ['distrito', 'año', 'acc_bici_total', 'acc_bici_leves', 'acc_bici_graves', 'acc_bici_mortales', 'acc_peatones_total', 'acc_peatones_leves', 'acc_peatones_graves', 'acc_peatones_mortales', 'poblacion', 'n_paradas_emt', 'km_carril_bici', 'acc_vru_total', 'acc_vru_leves', 'acc_vru_graves', 'acc_vru_mortales', 'acc_ponderado']


**Celda 23 — Añadir área por distrito y guardar versión 2**

In [27]:
# Calculamos área de cada distrito en km²
area_distritos = distritos.copy()
area_distritos['area_km2'] = (area_distritos.geometry.area / 1_000_000).round(2)
area_distritos['NOMDIS'] = area_distritos['NOMDIS'].str.replace(' - ', '-', regex=False)
area_distritos = area_distritos[['NOMDIS', 'area_km2']].rename(columns={'NOMDIS': 'distrito'})

# Creamos tabla_v2 sin modificar la tabla original
tabla_v2 = tabla.merge(area_distritos, on='distrito', how='left')

print(tabla_v2[['distrito', 'area_km2']].drop_duplicates().sort_values('distrito'))
print(f'\nNulos en area_km2: {tabla_v2["area_km2"].isnull().sum()}')

# Guardamos versión 2 con área
ruta_output_v2 = '/content/drive/MyDrive/TFM_Seguridad_Vial/outputs/tabla_maestra_v2.csv'
tabla_v2.to_csv(ruta_output_v2, index=False, encoding='utf-8-sig')
print('\n✅ tabla_maestra_v2.csv guardada con área por distrito')
print(f'✅ tabla_maestra.csv original intacta')

                distrito  area_km2
0             ARGANZUELA      6.46
6                BARAJAS     41.72
12           CARABANCHEL     14.05
18                CENTRO      5.23
24             CHAMARTÍN      9.18
30              CHAMBERÍ      4.68
36         CIUDAD LINEAL     11.43
42   FUENCARRAL-EL PARDO    237.84
48             HORTALEZA     27.62
54                LATINA     25.41
60       MONCLOA-ARAVACA     46.59
66             MORATALAZ      6.10
72    PUENTE DE VALLECAS     14.97
78                RETIRO      5.47
84             SALAMANCA      5.39
90   SAN BLAS-CANILLEJAS     22.29
96                TETUÁN      5.37
102                USERA      7.78
108            VICÁLVARO     34.84
114    VILLA DE VALLECAS     51.41
120           VILLAVERDE     20.19

Nulos en area_km2: 0

✅ tabla_maestra_v2.csv guardada con área por distrito
✅ tabla_maestra.csv original intacta


**Celda 24 — Agregar accidentes de bici por distrito y mes**

In [28]:
# Celda 24 corregida — Accidentes bici por distrito, año y mes

bici_m = pd.concat([
    pd.read_csv(os.path.join(BASE, 'accidentes_bici/raw', f), sep=';', encoding='utf-8-sig')
    for f in ficheros_bici
], ignore_index=True)

bici_m = bici_m.rename(columns={'tipo_vehículo': 'tipo_vehiculo', 'tipo_lesividad': 'lesividad'})
bici_m = bici_m.loc[:, ~bici_m.columns.duplicated()]
bici_m['fecha_dt'] = pd.to_datetime(bici_m['fecha'], dayfirst=True)
bici_m['año'] = bici_m['fecha_dt'].dt.year
bici_m['mes'] = bici_m['fecha_dt'].dt.month
bici_m['gravedad'] = bici_m['cod_lesividad'].map(mapa_gravedad).fillna('sin_asistencia')

# Gravedad máxima por expediente, distrito, año y mes
bici_m['gravedad_num'] = bici_m['gravedad'].map(jerarquia_gravedad).fillna(0)
grav_max_bici_m = (
    bici_m.groupby(['distrito', 'año', 'mes', 'num_expediente'])['gravedad_num']
    .max().reset_index()
)
grav_max_bici_m['gravedad_final'] = grav_max_bici_m['gravedad_num'].map(jerarquia_inversa)

def agregar_mensual(df_grav, grupo):
    base = df_grav.groupby(['distrito','año','mes'])['num_expediente'].nunique().reset_index().rename(columns={'num_expediente': f'acc_{grupo}_total'})
    leves   = df_grav[df_grav['gravedad_final']=='leve'].groupby(['distrito','año','mes'])['num_expediente'].nunique().reset_index().rename(columns={'num_expediente': f'acc_{grupo}_leves'})
    graves  = df_grav[df_grav['gravedad_final']=='grave'].groupby(['distrito','año','mes'])['num_expediente'].nunique().reset_index().rename(columns={'num_expediente': f'acc_{grupo}_graves'})
    mortales= df_grav[df_grav['gravedad_final']=='fallecido'].groupby(['distrito','año','mes'])['num_expediente'].nunique().reset_index().rename(columns={'num_expediente': f'acc_{grupo}_mortales'})
    r = base.merge(leves, on=['distrito','año','mes'], how='left')
    r = r.merge(graves,   on=['distrito','año','mes'], how='left')
    r = r.merge(mortales, on=['distrito','año','mes'], how='left')
    cols = [f'acc_{grupo}_leves', f'acc_{grupo}_graves', f'acc_{grupo}_mortales']
    r[cols] = r[cols].fillna(0).astype(int)
    return r

acc_bici_m = agregar_mensual(grav_max_bici_m, 'bici')
print(f'Filas accidentes bici mensual: {len(acc_bici_m)}')
print(acc_bici_m.head(3).to_string())

Filas accidentes bici mensual: 1544
     distrito   año  mes  acc_bici_total  acc_bici_leves  acc_bici_graves  acc_bici_mortales
0  ARGANZUELA  2019    1               3               2                0                  0
1  ARGANZUELA  2019    2               2               2                0                  0
2  ARGANZUELA  2019    3               4               2                1                  0


**Celda 25 — Agregar accidentes de peatones por distrito y mes**

In [29]:
# Celda 25 corregida — Accidentes peatones por distrito, año y mes

general_m = pd.concat([
    pd.read_csv(os.path.join(BASE, 'accidentes_general/raw', f), sep=';', encoding='utf-8-sig')
    for f in ficheros_general
], ignore_index=True)

general_m = general_m.rename(columns={'tipo_vehículo': 'tipo_vehiculo', 'tipo_lesividad': 'lesividad'})
general_m = general_m.loc[:, ~general_m.columns.duplicated()]
general_m['fecha_dt'] = pd.to_datetime(general_m['fecha'], dayfirst=True)
general_m['año'] = general_m['fecha_dt'].dt.year
general_m['mes'] = general_m['fecha_dt'].dt.month

peatones_m = general_m[general_m['tipo_persona'] == 'Peatón'].copy()
peatones_m['gravedad'] = peatones_m['cod_lesividad'].map(mapa_gravedad).fillna('sin_asistencia')
peatones_m['gravedad_num'] = peatones_m['gravedad'].map(jerarquia_gravedad).fillna(0)

grav_max_peat_m = (
    peatones_m.groupby(['distrito', 'año', 'mes', 'num_expediente'])['gravedad_num']
    .max().reset_index()
)
grav_max_peat_m['gravedad_final'] = grav_max_peat_m['gravedad_num'].map(jerarquia_inversa)

acc_peatones_m = agregar_mensual(grav_max_peat_m, 'peatones')
print(f'Filas accidentes peatones mensual: {len(acc_peatones_m)}')
print(acc_peatones_m.head(3).to_string())

Filas accidentes peatones mensual: 1707
     distrito   año  mes  acc_peatones_total  acc_peatones_leves  acc_peatones_graves  acc_peatones_mortales
0  ARGANZUELA  2019    1                  10                   9                    0                      0
1  ARGANZUELA  2019    2                   8                   6                    1                      0
2  ARGANZUELA  2019    3                   6                   4                    1                      1


**Celda 26 — Construir tabla maestra mensual**

In [30]:
# Celda 26 — Construir tabla maestra mensual

import itertools

distritos_lista = sorted(
    distritos['NOMDIS']
    .str.strip()
    .str.upper()
    .str.replace(' - ', '-', regex=False)
    .unique()
)
años_lista  = list(range(2019, 2025))
meses_lista = list(range(1, 13))

tabla_m = pd.DataFrame(
    list(itertools.product(distritos_lista, años_lista, meses_lista)),
    columns=['distrito', 'año', 'mes']
)
print(f'Combinaciones totales: {len(tabla_m)}')  # debe ser 1512

# Unimos bici y peatones por separado
tabla_m = tabla_m.merge(acc_bici_m,     on=['distrito', 'año', 'mes'], how='left')
tabla_m = tabla_m.merge(acc_peatones_m, on=['distrito', 'año', 'mes'], how='left')

cols_acc = [
    'acc_bici_total',     'acc_bici_leves',     'acc_bici_graves',     'acc_bici_mortales',
    'acc_peatones_total', 'acc_peatones_leves',  'acc_peatones_graves', 'acc_peatones_mortales'
]
tabla_m[cols_acc] = tabla_m[cols_acc].fillna(0).astype(int)

# ── VRU mensual sin doble conteo ─────────────────────────────────────────
orden_gravedad   = {'desconocido': 0, 'sin_asistencia': 1, 'leve': 2, 'grave': 3, 'fallecido': 4}
rank_a_gravedad  = {v: k for k, v in orden_gravedad.items()}

bici_m['gravedad_rank']     = bici_m['gravedad'].map(orden_gravedad)
peatones_m['gravedad_rank'] = peatones_m['gravedad'].map(orden_gravedad)

vru_m = pd.concat([
    bici_m.assign(origen='bici'),
    peatones_m.assign(origen='peaton')
], ignore_index=True)

vru_unicos_m = (
    vru_m.groupby(['distrito', 'año', 'mes', 'num_expediente'], as_index=False)
         .agg(gravedad_rank=('gravedad_rank', 'max'))
)
vru_unicos_m['gravedad_max'] = vru_unicos_m['gravedad_rank'].map(rank_a_gravedad)

acc_vru_m = (
    vru_unicos_m
    .groupby(['distrito', 'año', 'mes'])
    .agg(
        acc_vru_total   =('num_expediente', 'nunique'),
        acc_vru_leves   =('gravedad_max', lambda x: (x == 'leve').sum()),
        acc_vru_graves  =('gravedad_max', lambda x: (x == 'grave').sum()),
        acc_vru_mortales=('gravedad_max', lambda x: (x == 'fallecido').sum())
    )
    .reset_index()
)
acc_vru_m['acc_ponderado'] = (
    acc_vru_m['acc_vru_leves']    * 1 +
    acc_vru_m['acc_vru_graves']   * 7 +
    acc_vru_m['acc_vru_mortales'] * 16
)

tabla_m = tabla_m.merge(
    acc_vru_m[['distrito', 'año', 'mes',
               'acc_vru_total', 'acc_vru_leves', 'acc_vru_graves',
               'acc_vru_mortales', 'acc_ponderado']],
    on=['distrito', 'año', 'mes'], how='left'
)
cols_vru = ['acc_vru_total', 'acc_vru_leves', 'acc_vru_graves',
            'acc_vru_mortales', 'acc_ponderado']
tabla_m[cols_vru] = tabla_m[cols_vru].fillna(0)
print("✅ VRU mensual sin doble conteo")

# Unimos variables estructurales
tabla_anual = pd.read_csv(
    '/content/drive/MyDrive/TFM_Seguridad_Vial/outputs/tabla_maestra_v2.csv'
)
tabla_m = tabla_m.merge(
    tabla_anual[['distrito', 'año', 'poblacion', 'n_paradas_emt',
                 'km_carril_bici', 'area_km2']],
    on=['distrito', 'año'], how='left'
)

# Resumen
print(f'Forma tabla mensual: {tabla_m.shape}')
print(f'Nulos totales: {tabla_m.isnull().sum().sum()}')
print(f'Columnas: {tabla_m.columns.tolist()}')

conteo = tabla_m.groupby('distrito')['mes'].count()
if (conteo != 72).any():
    print(f'\n⚠️ Algunos distritos no tienen 72 meses')
    print(conteo[conteo != 72])
else:
    print(f'\n✅ Todos los distritos tienen exactamente 72 meses')

Combinaciones totales: 1512
✅ VRU mensual sin doble conteo
Forma tabla mensual: (1512, 20)
Nulos totales: 0
Columnas: ['distrito', 'año', 'mes', 'acc_bici_total', 'acc_bici_leves', 'acc_bici_graves', 'acc_bici_mortales', 'acc_peatones_total', 'acc_peatones_leves', 'acc_peatones_graves', 'acc_peatones_mortales', 'acc_vru_total', 'acc_vru_leves', 'acc_vru_graves', 'acc_vru_mortales', 'acc_ponderado', 'poblacion', 'n_paradas_emt', 'km_carril_bici', 'area_km2']

✅ Todos los distritos tienen exactamente 72 meses


**Celda 27 — Guardar tabla maestra mensual**

In [31]:
ruta_mensual = '/content/drive/MyDrive/TFM_Seguridad_Vial/outputs/tabla_maestra_mensual.csv'
tabla_m.to_csv(ruta_mensual, index=False, encoding='utf-8-sig')
print('✅ tabla_maestra_mensual.csv guardada correctamente')
print(f'Forma: {tabla_m.shape}')
print(f'Columnas: {tabla_m.columns.tolist()}')

✅ tabla_maestra_mensual.csv guardada correctamente
Forma: (1512, 20)
Columnas: ['distrito', 'año', 'mes', 'acc_bici_total', 'acc_bici_leves', 'acc_bici_graves', 'acc_bici_mortales', 'acc_peatones_total', 'acc_peatones_leves', 'acc_peatones_graves', 'acc_peatones_mortales', 'acc_vru_total', 'acc_vru_leves', 'acc_vru_graves', 'acc_vru_mortales', 'acc_ponderado', 'poblacion', 'n_paradas_emt', 'km_carril_bici', 'area_km2']


**Celda 28 — Extraer % población por edad (mayores 65 y menores 15) por distrito**

In [32]:
# Cargamos el panel de indicadores
ruta_panel = os.path.join(BASE, 'poblacion_distrito/raw/indicadores_distritos_acumulado.csv')
panel = pd.read_csv(ruta_panel, sep=';', encoding='utf-8-sig')

# Convertimos valor_indicador a numérico
panel['valor_indicador'] = pd.to_numeric(
    panel['valor_indicador'].astype(str).str.replace(',', '.'), errors='coerce'
)

# Filtramos solo nivel distrito (cod_barrio nulo) y años 2019-2024
panel_dist = panel[
    (panel['cod_barrio'].isna()) &
    (panel['año'].between(2019, 2024)) &
    (panel['cod_distrito'] > 0)
].copy()

# Extraemos % mayores de 65
mayores65 = panel_dist[
    panel_dist['indicador_completo'] == 'Porcentaje de envejecimiento (población mayor de 65 años/población total)'
][['cod_distrito', 'distrito', 'año', 'valor_indicador']].copy()
mayores65.columns = ['cod_distrito', 'distrito', 'año', 'pct_mayores_65']

# Extraemos población 0-14 años
menores15_abs = panel_dist[
    panel_dist['indicador_completo'] == 'Población de 0 a 14 años'
][['cod_distrito', 'distrito', 'año', 'valor_indicador']].copy()
menores15_abs.columns = ['cod_distrito', 'distrito', 'año', 'pob_menores_15']

# Extraemos población total
pob_total = panel_dist[
    panel_dist['indicador_completo'] == 'Número habitantes'
][['cod_distrito', 'año', 'valor_indicador']].copy()
pob_total.columns = ['cod_distrito', 'año', 'pob_total']

# Calculamos % menores de 15
menores15 = menores15_abs.merge(pob_total, on=['cod_distrito', 'año'], how='left')
menores15['pct_menores_15'] = (menores15['pob_menores_15'] / menores15['pob_total'] * 100).round(2)
menores15 = menores15[['cod_distrito', 'distrito', 'año', 'pct_menores_15']]

# Unimos ambos
edad = mayores65.merge(menores15, on=['cod_distrito', 'distrito', 'año'], how='outer')
edad['distrito'] = edad['distrito'].str.strip().str.upper()

print(edad.head(10))
print(f'\nForma: {edad.shape}')
print(f'Años: {sorted(edad["año"].unique())}')
print(f'\nNulos:\n{edad.isnull().sum()}')

   cod_distrito    distrito   año  pct_mayores_65  pct_menores_15
0           1.0      CENTRO  2019       16.050000            7.82
1           1.0      CENTRO  2020       16.000000             NaN
2           1.0      CENTRO  2021       15.550000            7.04
3           1.0      CENTRO  2022       15.690000            6.91
4           1.0      CENTRO  2023       15.870000            6.73
5           1.0      CENTRO  2024       15.675568            6.51
6           2.0  ARGANZUELA  2019       19.270000           11.82
7           2.0  ARGANZUELA  2020       20.000000             NaN
8           2.0  ARGANZUELA  2021       19.830000           11.25
9           2.0  ARGANZUELA  2022       20.450000           11.06

Forma: (138, 5)
Años: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]

Nulos:
cod_distrito       0
distrito           0
año                0
pct_mayores_65     0
pct_menores_15    25
dtype: int64


**Celda 29 — Limpiar datos de edad**

In [33]:
# Verificamos qué años tienen nulos en pct_menores_15
print("Nulos por año:")
print(edad.groupby('año')['pct_menores_15'].apply(lambda x: x.isnull().sum()))

# Verificamos duplicados
print(f"\nDuplicados: {edad.duplicated(subset=['cod_distrito', 'año']).sum()}")
print(f"Distritos únicos: {edad['distrito'].nunique()}")

# Imputamos 2020 con media de 2019 y 2021
for dist in edad['distrito'].unique():
    mask_2020 = (edad['distrito'] == dist) & (edad['año'] == 2020)
    val_2019 = edad[(edad['distrito'] == dist) & (edad['año'] == 2019)]['pct_menores_15'].values
    val_2021 = edad[(edad['distrito'] == dist) & (edad['año'] == 2021)]['pct_menores_15'].values
    if len(val_2019) > 0 and len(val_2021) > 0:
        edad.loc[mask_2020, 'pct_menores_15'] = round((val_2019[0] + val_2021[0]) / 2, 2)

# Eliminamos duplicados si los hay
edad = edad.drop_duplicates(subset=['cod_distrito', 'año']).reset_index(drop=True)

print(f"\nForma final: {edad.shape}")
print(f"Nulos restantes:\n{edad.isnull().sum()}")

Nulos por año:
año
2019     0
2020    21
2021     0
2022     0
2023     0
2024     4
Name: pct_menores_15, dtype: int64

Duplicados: 12
Distritos únicos: 21

Forma final: (126, 5)
Nulos restantes:
cod_distrito      0
distrito          0
año               0
pct_mayores_65    0
pct_menores_15    4
dtype: int64


**Celda 30 — Imputar nulos de 2024**

In [34]:
# Verificamos qué distritos tienen nulo en 2024
nulos_2024 = edad[(edad['año'] == 2024) & (edad['pct_menores_15'].isna())]['distrito'].tolist()
print(f"Distritos con nulo en 2024: {nulos_2024}")

# Imputamos con el valor de 2023 para esos distritos
for dist in nulos_2024:
    mask_2024 = (edad['distrito'] == dist) & (edad['año'] == 2024)
    val_2023 = edad[(edad['distrito'] == dist) & (edad['año'] == 2023)]['pct_menores_15'].values
    if len(val_2023) > 0:
        edad.loc[mask_2024, 'pct_menores_15'] = val_2023[0]

print(f"Nulos restantes: {edad['pct_menores_15'].isnull().sum()}")
print(f"\nForma final: {edad.shape}")
print(edad.groupby('año')['pct_menores_15'].mean().round(2))

Distritos con nulo en 2024: ['RETIRO', 'SALAMANCA', 'CHAMARTÍN', 'TETUÁN']
Nulos restantes: 0

Forma final: (126, 5)
año
2019    13.46
2020    13.17
2021    12.87
2022    12.71
2023    12.40
2024    12.12
Name: pct_menores_15, dtype: float64


**Celda 31 — Añadir edad a las tablas maestras**

In [35]:
# Añadimos a tabla_maestra_v2
tabla_v2 = pd.read_csv('/content/drive/MyDrive/TFM_Seguridad_Vial/outputs/tabla_maestra_v2.csv')
tabla_v3 = tabla_v2.merge(
    edad[['distrito', 'año', 'pct_mayores_65', 'pct_menores_15']],
    on=['distrito', 'año'], how='left'
)
print(f'Forma tabla_v3: {tabla_v3.shape}')
print(f'Nulos: {tabla_v3[["pct_mayores_65","pct_menores_15"]].isnull().sum().sum()}')

# Guardamos tabla_maestra_v3
tabla_v3.to_csv('/content/drive/MyDrive/TFM_Seguridad_Vial/outputs/tabla_maestra_v3.csv',
                index=False, encoding='utf-8-sig')
print('✅ tabla_maestra_v3.csv guardada')

# Añadimos a tabla_maestra_mensual
tabla_m = pd.read_csv('/content/drive/MyDrive/TFM_Seguridad_Vial/outputs/tabla_maestra_mensual.csv')
tabla_m_v2 = tabla_m.merge(
    edad[['distrito', 'año', 'pct_mayores_65', 'pct_menores_15']],
    on=['distrito', 'año'], how='left'
)
print(f'\nForma tabla_mensual_v2: {tabla_m_v2.shape}')
print(f'Nulos: {tabla_m_v2[["pct_mayores_65","pct_menores_15"]].isnull().sum().sum()}')

# Guardamos tabla_maestra_mensual_v2
tabla_m_v2.to_csv('/content/drive/MyDrive/TFM_Seguridad_Vial/outputs/tabla_maestra_mensual_v2.csv',
                   index=False, encoding='utf-8-sig')
print('✅ tabla_maestra_mensual_v2.csv guardada')

Forma tabla_v3: (126, 21)
Nulos: 0
✅ tabla_maestra_v3.csv guardada

Forma tabla_mensual_v2: (1512, 22)
Nulos: 0
✅ tabla_maestra_mensual_v2.csv guardada


**Celda 32 — Redondear porcentajes de edad**

In [36]:
# Redondeamos a 2 decimales en ambas tablas
tabla_v3['pct_mayores_65'] = tabla_v3['pct_mayores_65'].round(2)
tabla_v3['pct_menores_15'] = tabla_v3['pct_menores_15'].round(2)

tabla_m_v2['pct_mayores_65'] = tabla_m_v2['pct_mayores_65'].round(2)
tabla_m_v2['pct_menores_15'] = tabla_m_v2['pct_menores_15'].round(2)

# Guardamos de nuevo
tabla_v3.to_csv('/content/drive/MyDrive/TFM_Seguridad_Vial/outputs/tabla_maestra_v3.csv',
                index=False, encoding='utf-8-sig')
tabla_m_v2.to_csv('/content/drive/MyDrive/TFM_Seguridad_Vial/outputs/tabla_maestra_mensual_v2.csv',
                   index=False, encoding='utf-8-sig')

print('✅ Tablas actualizadas con redondeo a 2 decimales')
print(tabla_v3[['distrito', 'año', 'pct_mayores_65', 'pct_menores_15']].head(10))

✅ Tablas actualizadas con redondeo a 2 decimales
     distrito   año  pct_mayores_65  pct_menores_15
0  ARGANZUELA  2019           19.27           11.82
1  ARGANZUELA  2020           20.00           11.54
2  ARGANZUELA  2021           19.83           11.25
3  ARGANZUELA  2022           20.45           11.06
4  ARGANZUELA  2023           20.87           10.73
5  ARGANZUELA  2024           21.07           10.40
6     BARAJAS  2019           17.77           17.21
7     BARAJAS  2020           18.00           16.80
8     BARAJAS  2021           17.89           16.40
9     BARAJAS  2022           18.77           15.57


**Celda 33 — Instalar osmnx y extraer % red viaria ≤30km/h**

In [37]:
# Instalamos osmnx (puede tardar 1-2 minutos)
!pip install osmnx --quiet

import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np

print('osmnx instalado correctamente')
print(f'Versión osmnx: {ox.__version__}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 4.6 MB/s eta 0:00:00
osmnx instalado correctamente
Versión osmnx: 2.1.0


**Celda 34 — Descargar red viaria y calcular % ≤30km/h por distrito**

In [38]:
# Cargamos el shapefile de distritos que ya tenemos en Drive
ruta_barrios = os.path.join(BASE, 'distritos/raw/BARRIOS.shp')
barrios = gpd.read_file(ruta_barrios)

# Creamos capa de distritos
distritos_geo = barrios.dissolve(by='CODDIS', aggfunc='first')[['NOMDIS', 'geometry']].reset_index()
distritos_geo['NOMDIS'] = distritos_geo['NOMDIS'].str.strip().str.upper().str.replace(' - ', '-', regex=False)
distritos_geo = distritos_geo.to_crs('EPSG:4326')  # osmnx necesita WGS84

print(f'Distritos cargados: {len(distritos_geo)}')
print('Descargando red viaria de Madrid... (puede tardar 5-10 minutos)')

# Descargamos la red viaria de Madrid completa
G = ox.graph_from_place('Madrid, Spain', network_type='drive')
edges = ox.graph_to_gdfs(G, nodes=False)

print(f'Tramos de red viaria descargados: {len(edges)}')
print(f'Columnas disponibles: {edges.columns.tolist()}')
print(f'\nValores únicos de maxspeed (muestra):')
print(edges['maxspeed'].value_counts().head(10) if 'maxspeed' in edges.columns else 'No hay columna maxspeed')

Distritos cargados: 21
Descargando red viaria de Madrid... (puede tardar 5-10 minutos)
Tramos de red viaria descargados: 61801
Columnas disponibles: ['osmid', 'highway', 'lanes', 'maxspeed', 'name', 'oneway', 'reversed', 'length', 'geometry', 'junction', 'bridge', 'width', 'access', 'ref', 'tunnel', 'est_width']

Valores únicos de maxspeed (muestra):
maxspeed
50          12306
30           8781
40           2156
20            965
90            216
70            207
100           200
60            180
80            180
[30, 50]       74
Name: count, dtype: int64


**Celda 35 — Calcular % red viaria ≤30km/h por distrito**

In [39]:
# Convertimos edges a GeoDataFrame con CRS correcto
edges_geo = edges.copy()
edges_geo = edges_geo.set_crs('EPSG:4326').to_crs('EPSG:25830')

# Limpiamos maxspeed: extraemos el valor mínimo si es lista
def limpiar_velocidad(v):
    if isinstance(v, list):
        try:
            return min([int(x) for x in v])
        except:
            return np.nan
    try:
        return float(v)
    except:
        return np.nan

edges_geo['vel_num'] = edges_geo['maxspeed'].apply(limpiar_velocidad)

print(f'Tramos con velocidad conocida: {edges_geo["vel_num"].notna().sum()}')
print(f'Tramos sin velocidad: {edges_geo["vel_num"].isna().sum()}')
print(f'\nDistribución de velocidades:')
print(edges_geo['vel_num'].value_counts().sort_index())

Tramos con velocidad conocida: 25668
Tramos sin velocidad: 36133

Distribución de velocidades:
vel_num
10.0         1
15.0        12
20.0      1039
30.0      8876
40.0      2261
50.0     12352
60.0       206
70.0       230
80.0       204
90.0       225
100.0      205
120.0       57
Name: count, dtype: int64


**Celda 36 — Imputar velocidad por tipo de vía y calcular % ≤30km/h por distrito**

In [40]:
# Tabla de velocidades implícitas
velocidad_implicita = {
    'living_street': 10, 'pedestrian': 10,
    'residential': 30, 'unclassified': 30,
    'tertiary': 30, 'tertiary_link': 30,
    'secondary': 50, 'secondary_link': 50,
    'primary': 50, 'primary_link': 50,
    'trunk': 80, 'trunk_link': 80,
    'motorway': 120, 'motorway_link': 120
}

edges_geo['vel_final'] = edges_geo['vel_num'].copy()
mask_nulo = edges_geo['vel_final'].isna()

def limpiar_highway(h):
    if isinstance(h, list):
        return h[0]
    return h

edges_geo['highway_clean'] = edges_geo['highway'].apply(limpiar_highway)
edges_geo.loc[mask_nulo, 'vel_final'] = edges_geo.loc[mask_nulo, 'highway_clean'].map(velocidad_implicita)

print(f'Tramos con velocidad tras imputación: {edges_geo["vel_final"].notna().sum()}')
print(f'Tramos sin velocidad restantes: {edges_geo["vel_final"].isna().sum()}')

# ── CORRECCIÓN: usar overlay en lugar de sjoin ────────────────────────────────
# sjoin asigna el tramo completo a todos los distritos que toca (sobreconteo)
# overlay recorta cada tramo por el límite del distrito y recalcula la longitud

distritos_25830 = distritos_geo.to_crs('EPSG:25830')

edges_validos = edges_geo[edges_geo['vel_final'].notna()][['vel_final', 'geometry']].copy()

# Recortamos los tramos por los límites de cada distrito
edges_clip = gpd.overlay(edges_validos, distritos_25830[['NOMDIS', 'geometry']], how='intersection')

# Recalculamos la longitud real dentro de cada distrito (en metros)
edges_clip['length_clip'] = edges_clip.geometry.length

print(f'\nTramos tras overlay: {len(edges_clip)}')
print(f'Nulos en vel_final tras overlay: {edges_clip["vel_final"].isna().sum()}')

# Calculamos % de longitud ≤30km/h por distrito usando longitud recortada
pct_30 = (
    edges_clip.groupby('NOMDIS')[['vel_final', 'length_clip']]
    .apply(lambda g: round(
        g[g['vel_final'] <= 30]['length_clip'].sum() / g['length_clip'].sum() * 100, 2
    ), include_groups=False)
    .reset_index()
    .rename(columns={'NOMDIS': 'distrito', 0: 'pct_red_30kmh'})
)

pct_30['distrito'] = pct_30['distrito'].str.strip().str.upper().str.replace(' - ', '-', regex=False)

print('\npct_red_30kmh por distrito:')
print(pct_30.sort_values('pct_red_30kmh', ascending=False).to_string())

Tramos con velocidad tras imputación: 61781
Tramos sin velocidad restantes: 20

Tramos tras overlay: 62823
Nulos en vel_final tras overlay: 0

pct_red_30kmh por distrito:
               distrito  pct_red_30kmh
3                CENTRO          73.36
17                USERA          71.99
2           CARABANCHEL          71.63
12   PUENTE DE VALLECAS          70.89
16               TETUÁN          69.27
20           VILLAVERDE          66.89
9                LATINA          64.37
1               BARAJAS          63.26
7   FUENCARRAL-EL PARDO          60.65
15  SAN BLAS-CANILLEJAS          59.99
10      MONCLOA-ARAVACA          59.90
11            MORATALAZ          58.18
19    VILLA DE VALLECAS          57.42
14            SALAMANCA          54.05
5              CHAMBERÍ          53.02
8             HORTALEZA          50.73
13               RETIRO          49.39
4             CHAMARTÍN          46.02
18            VICÁLVARO          44.79
0            ARGANZUELA          41.36
6         

In [41]:
# Celda 36A — Velocidad media ponderada por distrito (con longitud recortada)

vel_media = (
    edges_clip.groupby('NOMDIS')
    .apply(lambda g: round(
        (g['vel_final'] * g['length_clip']).sum() / g['length_clip'].sum(), 2
    ), include_groups=False)
    .reset_index()
    .rename(columns={'NOMDIS': 'distrito', 0: 'vel_media_vial'})
)

vel_media['distrito'] = (
    vel_media['distrito']
    .str.strip()
    .str.upper()
    .str.replace(' - ', '-', regex=False)
)

print(vel_media.sort_values('vel_media_vial', ascending=False).to_string())

               distrito  vel_media_vial
18            VICÁLVARO           55.91
19    VILLA DE VALLECAS           50.71
1               BARAJAS           47.78
11            MORATALAZ           45.67
8             HORTALEZA           45.22
0            ARGANZUELA           44.86
15  SAN BLAS-CANILLEJAS           44.83
6         CIUDAD LINEAL           44.56
7   FUENCARRAL-EL PARDO           44.00
10      MONCLOA-ARAVACA           43.97
13               RETIRO           43.35
20           VILLAVERDE           43.27
4             CHAMARTÍN           42.27
14            SALAMANCA           41.21
12   PUENTE DE VALLECAS           40.06
9                LATINA           39.78
5              CHAMBERÍ           38.93
2           CARABANCHEL           37.63
17                USERA           37.57
16               TETUÁN           35.81
3                CENTRO           32.59


In [42]:
# Celda 36B — Extraer semáforos peatonales por distrito desde OSM

# En osmnx ≥ 2.0, geometries_from_place se reemplaza por features_from_place
semaforos = ox.features_from_place(
    'Madrid, Spain',
    tags={'highway': 'crossing', 'crossing': 'traffic_signals'}
)

print(f'Elementos descargados: {len(semaforos)}')
print(f'Tipos de geometría: {semaforos.geometry.geom_type.value_counts().to_dict()}')

Elementos descargados: 39412
Tipos de geometría: {'Point': 36631, 'LineString': 2781}


In [43]:
# Celda 36C — Convertir a puntos y asignar a distritos

# Nos quedamos solo con puntos (los cruces son nodos, no polígonos)
semaforos_pts = semaforos[semaforos.geometry.geom_type == 'Point'].copy()
semaforos_pts = semaforos_pts[['geometry']].reset_index(drop=True)
semaforos_pts = semaforos_pts.set_crs('EPSG:4326').to_crs('EPSG:25830')

print(f'Semáforos peatonales (puntos): {len(semaforos_pts)}')

# Spatial join con distritos
distritos_25830 = distritos_geo.to_crs('EPSG:25830')
semaforos_distrito = gpd.sjoin(
    semaforos_pts,
    distritos_25830[['NOMDIS', 'geometry']],
    how='left',
    predicate='within'
)

# Contamos semáforos por distrito
n_semaforos = (
    semaforos_distrito.groupby('NOMDIS')
    .size()
    .reset_index()
    .rename(columns={'NOMDIS': 'distrito', 0: 'n_semaforos_peatonales'})
)

n_semaforos['distrito'] = (
    n_semaforos['distrito']
    .str.strip()
    .str.upper()
    .str.replace(' - ', '-', regex=False)
)

# Calculamos densidad: semáforos por km²
area_distritos = distritos_25830[['NOMDIS', 'geometry']].copy()
area_distritos['area_km2'] = (area_distritos.geometry.area / 1_000_000).round(2)
area_distritos['NOMDIS'] = (
    area_distritos['NOMDIS']
    .str.strip()
    .str.upper()
    .str.replace(' - ', '-', regex=False)
)
area_distritos = area_distritos[['NOMDIS', 'area_km2']].rename(columns={'NOMDIS': 'distrito'})

n_semaforos = n_semaforos.merge(area_distritos, on='distrito', how='left')
n_semaforos['semaforos_km2'] = (
    n_semaforos['n_semaforos_peatonales'] / n_semaforos['area_km2']
).round(2)

print(n_semaforos.sort_values('semaforos_km2', ascending=False).to_string())

# Verificación
sin_distrito = semaforos_distrito['NOMDIS'].isna().sum()
if sin_distrito > 0:
    print(f'\n⚠️ {sin_distrito} semáforos sin distrito asignado')
else:
    print('\n✅ Todos los semáforos asignados a un distrito')

Semáforos peatonales (puntos): 36631
               distrito  n_semaforos_peatonales  area_km2  semaforos_km2
16               TETUÁN                    1689      5.37         314.53
5              CHAMBERÍ                    1016      4.68         217.09
0            ARGANZUELA                    1392      6.46         215.48
14            SALAMANCA                    1133      5.39         210.20
3                CENTRO                     936      5.23         178.97
13               RETIRO                     941      5.47         172.03
6         CIUDAD LINEAL                    1951     11.43         170.69
17                USERA                    1293      7.78         166.20
4             CHAMARTÍN                    1308      9.18         142.48
2           CARABANCHEL                    1963     14.05         139.72
8             HORTALEZA                    3544     27.62         128.31
12   PUENTE DE VALLECAS                    1675     14.97         111.89
20           V

**Celda 37 — Añadir % red 30kmh a las tablas maestras**

In [44]:
# Es un dato estático (no varía por año), lo añadimos igual que km_carril_bici

# Tabla anual v3
tabla_v3 = pd.read_csv('/content/drive/MyDrive/TFM_Seguridad_Vial/outputs/tabla_maestra_v3.csv')

# Añadimos pct_red_30kmh
tabla_v4 = tabla_v3.merge(pct_30, on='distrito', how='left')
print(f'Nulos pct_red_30kmh: {tabla_v4["pct_red_30kmh"].isnull().sum()}')

# Añadimos semáforos peatonales
tabla_v4 = tabla_v4.merge(
    n_semaforos[['distrito', 'n_semaforos_peatonales', 'semaforos_km2']],
    on='distrito', how='left'
)
print(f'Nulos semaforos_km2: {tabla_v4["semaforos_km2"].isnull().sum()}')

# Añadimos velocidad media vial
tabla_v4 = tabla_v4.merge(vel_media, on='distrito', how='left')
print(f'Nulos vel_media_vial: {tabla_v4["vel_media_vial"].isnull().sum()}')
print(f'Forma tabla_v4: {tabla_v4.shape}')

tabla_v4.to_csv('/content/drive/MyDrive/TFM_Seguridad_Vial/outputs/tabla_maestra_v4.csv',
                index=False, encoding='utf-8-sig')
print('✅ tabla_maestra_v4.csv guardada')

# Tabla mensual v2
tabla_m_v2 = pd.read_csv('/content/drive/MyDrive/TFM_Seguridad_Vial/outputs/tabla_maestra_mensual_v2.csv')

# Añadimos pct_red_30kmh
tabla_m_v3 = tabla_m_v2.merge(pct_30, on='distrito', how='left')

# Añadimos semáforos peatonales
tabla_m_v3 = tabla_m_v3.merge(
    n_semaforos[['distrito', 'n_semaforos_peatonales', 'semaforos_km2']],
    on='distrito', how='left'
)

# Añadimos velocidad media vial
tabla_m_v3 = tabla_m_v3.merge(vel_media, on='distrito', how='left')

print(f'\nForma tabla_mensual_v3: {tabla_m_v3.shape}')
print(f'Nulos pct_red_30kmh: {tabla_m_v3["pct_red_30kmh"].isnull().sum()}')
print(f'Nulos semaforos_km2: {tabla_m_v3["semaforos_km2"].isnull().sum()}')
print(f'Nulos vel_media_vial: {tabla_m_v3["vel_media_vial"].isnull().sum()}')

tabla_m_v3.to_csv('/content/drive/MyDrive/TFM_Seguridad_Vial/outputs/tabla_maestra_mensual_v3.csv',
                   index=False, encoding='utf-8-sig')
print('✅ tabla_maestra_mensual_v3.csv guardada')

# Resumen columnas finales
print(f'\nColumnas tabla anual: {tabla_v4.columns.tolist()}')
print(f'Columnas tabla mensual: {tabla_m_v3.columns.tolist()}')

Nulos pct_red_30kmh: 0
Nulos semaforos_km2: 0
Nulos vel_media_vial: 0
Forma tabla_v4: (126, 25)
✅ tabla_maestra_v4.csv guardada

Forma tabla_mensual_v3: (1512, 26)
Nulos pct_red_30kmh: 0
Nulos semaforos_km2: 0
Nulos vel_media_vial: 0
✅ tabla_maestra_mensual_v3.csv guardada

Columnas tabla anual: ['distrito', 'año', 'acc_bici_total', 'acc_bici_leves', 'acc_bici_graves', 'acc_bici_mortales', 'acc_peatones_total', 'acc_peatones_leves', 'acc_peatones_graves', 'acc_peatones_mortales', 'poblacion', 'n_paradas_emt', 'km_carril_bici', 'acc_vru_total', 'acc_vru_leves', 'acc_vru_graves', 'acc_vru_mortales', 'acc_ponderado', 'area_km2', 'pct_mayores_65', 'pct_menores_15', 'pct_red_30kmh', 'n_semaforos_peatonales', 'semaforos_km2', 'vel_media_vial']
Columnas tabla mensual: ['distrito', 'año', 'mes', 'acc_bici_total', 'acc_bici_leves', 'acc_bici_graves', 'acc_bici_mortales', 'acc_peatones_total', 'acc_peatones_leves', 'acc_peatones_graves', 'acc_peatones_mortales', 'acc_vru_total', 'acc_vru_leves',

# Bloque 6 — Estaciones BiciMAD por distrito

In [45]:
# Bloque 6 — Estaciones BiciMAD por distrito

# ── Celda A: Cargar estaciones BiciMAD desde GeoJSON público ──────────────────

import geopandas as gpd

url_bicimad = (
    "https://datos.emtmadrid.es/dataset/5fcc0945-2cbd-46c3-801a-6a83f4167c11"
    "/resource/105ce5df-793f-4e0a-a88e-5d3b3f024a5d"
    "/download/bikestationbicimad_geojson.geojson"
)

estaciones = gpd.read_file(url_bicimad)

print(f'Estaciones cargadas: {len(estaciones)}')
print(f'CRS: {estaciones.crs}')
print(f'Columnas: {estaciones.columns.tolist()}')
print(estaciones.head(3).to_string())

Estaciones cargadas: 631
CRS: EPSG:4326
Columnas: ['OBJECTID', 'Activate', 'Address', 'PosicionSTR', 'Ligth', 'Name', 'NoAvailable', 'number', 'TotalBases', 'State', 'POINT_X', 'POINT_Y', 'geometry']
   OBJECTID  Activate                       Address                     PosicionSTR  Ligth                    Name  NoAvailable number  TotalBases       State   POINT_X    POINT_Y                   geometry
0         1         1       Calle Miguel Moya nº 1,     -3.7056899999999997 40.4204      0        2 - Metro Callao            0      2          27  IN_SERVICE -3.705690  40.420400   POINT (-3.70569 40.4204)
1         2         1     Plaza Conde Surchill, 4 ,  -3.7072537000000003 40.4303223      1  3 - Plaza Conde Suchil            0      3          19  IN_SERVICE -3.707254  40.430322  POINT (-3.70725 40.43032)
2         3         1  Calle Manuela Malasaña nº 3,              -3.7025 40.4286259      0            4 - Malasaña            0      4          27  IN_SERVICE -3.702500  40.428626

In [46]:
# ── Celda B: Asignar estaciones a distritos mediante spatial join ─────────────

# Convertimos distritos a WGS84 para coincidir con BiciMAD
distritos_wgs84 = distritos.to_crs('EPSG:4326')
distritos_wgs84['NOMDIS'] = (
    distritos_wgs84['NOMDIS']
    .str.strip()
    .str.upper()
    .str.replace(' - ', '-', regex=False)
)

# Spatial join: qué estaciones caen en cada distrito
estaciones_distrito = gpd.sjoin(
    estaciones,
    distritos_wgs84[['NOMDIS', 'geometry']],
    how='left',
    predicate='within'
)

# Contamos estaciones por distrito
n_estaciones_bicimad = (
    estaciones_distrito.groupby('NOMDIS')
    .size()
    .reset_index()
    .rename(columns={'NOMDIS': 'distrito', 0: 'n_estaciones_bicimad'})
)

n_estaciones_bicimad['distrito'] = (
    n_estaciones_bicimad['distrito']
    .str.strip()
    .str.upper()
    .str.replace(' - ', '-', regex=False)
)

print(n_estaciones_bicimad.sort_values('n_estaciones_bicimad', ascending=False).to_string())
print(f'\nTotal distritos con estaciones: {len(n_estaciones_bicimad)}')
print(f'Total estaciones asignadas: {n_estaciones_bicimad["n_estaciones_bicimad"].sum()}')

# Verificación: estaciones sin distrito asignado
sin_distrito = estaciones_distrito['NOMDIS'].isna().sum()
if sin_distrito > 0:
    print(f'⚠️ {sin_distrito} estaciones sin distrito asignado (fuera del municipio)')
else:
    print('✅ Todas las estaciones asignadas a un distrito')

               distrito  n_estaciones_bicimad
3                CENTRO                    59
7   FUENCARRAL-EL PARDO                    44
14            SALAMANCA                    42
0            ARGANZUELA                    40
10      MONCLOA-ARAVACA                    39
4             CHAMARTÍN                    36
2           CARABANCHEL                    35
12   PUENTE DE VALLECAS                    31
8             HORTALEZA                    31
6         CIUDAD LINEAL                    31
9                LATINA                    31
15  SAN BLAS-CANILLEJAS                    30
13               RETIRO                    30
5              CHAMBERÍ                    29
16               TETUÁN                    23
20           VILLAVERDE                    20
19    VILLA DE VALLECAS                    19
18            VICÁLVARO                    18
1               BARAJAS                    16
17                USERA                    16
11            MORATALAZ           

In [47]:
# ── Celda C: Añadir a las tablas maestras ─────────────────────────────────────
# Es variable estática — mismo valor para todos los años

# Tabla anual
tabla_v4 = pd.read_csv(
    '/content/drive/MyDrive/TFM_Seguridad_Vial/outputs/tabla_maestra_v4.csv'
)
tabla_v5 = tabla_v4.merge(n_estaciones_bicimad, on='distrito', how='left')

# Verificación
print(f'Nulos n_estaciones_bicimad: {tabla_v5["n_estaciones_bicimad"].isnull().sum()}')
print(f'Forma tabla_v5: {tabla_v5.shape}')

# Si hay distritos sin estaciones BiciMAD, rellenamos con 0
tabla_v5['n_estaciones_bicimad'] = tabla_v5['n_estaciones_bicimad'].fillna(0).astype(int)

tabla_v5.to_csv(
    '/content/drive/MyDrive/TFM_Seguridad_Vial/outputs/tabla_maestra_v5.csv',
    index=False, encoding='utf-8-sig'
)
print('✅ tabla_maestra_v5.csv guardada con estaciones BiciMAD')
print(f'Columnas: {tabla_v5.columns.tolist()}')

Nulos n_estaciones_bicimad: 0
Forma tabla_v5: (126, 26)
✅ tabla_maestra_v5.csv guardada con estaciones BiciMAD
Columnas: ['distrito', 'año', 'acc_bici_total', 'acc_bici_leves', 'acc_bici_graves', 'acc_bici_mortales', 'acc_peatones_total', 'acc_peatones_leves', 'acc_peatones_graves', 'acc_peatones_mortales', 'poblacion', 'n_paradas_emt', 'km_carril_bici', 'acc_vru_total', 'acc_vru_leves', 'acc_vru_graves', 'acc_vru_mortales', 'acc_ponderado', 'area_km2', 'pct_mayores_65', 'pct_menores_15', 'pct_red_30kmh', 'n_semaforos_peatonales', 'semaforos_km2', 'vel_media_vial', 'n_estaciones_bicimad']


In [48]:
# Celda de validación — Accidentes 2025 por distrito (para Power BI)

accidentes_2025 = accidentes_bici[accidentes_bici['año'] == 2025].copy()
peatones_2025   = accidentes_peatones[accidentes_peatones['año'] == 2025].copy()

# Grid base con año incluido
validacion_2025 = pd.DataFrame({
    'distrito': distritos_lista,
    'año':      2025
})

# Mergeamos bici y peatones por separado (para mantener columnas individuales)
validacion_2025 = validacion_2025.merge(accidentes_2025, on=['distrito', 'año'], how='left')
validacion_2025 = validacion_2025.merge(peatones_2025,   on=['distrito', 'año'], how='left')

cols_acc = [
    'acc_bici_total',    'acc_bici_leves',    'acc_bici_graves',    'acc_bici_mortales',
    'acc_peatones_total','acc_peatones_leves', 'acc_peatones_graves','acc_peatones_mortales'
]
validacion_2025[cols_acc] = validacion_2025[cols_acc].fillna(0).astype(int)

# ── VRU sin doble conteo — desde accidentes_vru_unicos ───────────────────
# accidentes_vru_unicos incluye 2025 porque bici y peatones incluyen 2025
vru_2025 = accidentes_vru_unicos[accidentes_vru_unicos['año'] == 2025][
    ['distrito', 'año', 'acc_vru_total', 'acc_vru_leves',
     'acc_vru_graves', 'acc_vru_mortales', 'acc_ponderado']
].copy()

validacion_2025 = validacion_2025.merge(vru_2025, on=['distrito', 'año'], how='left')

cols_vru = ['acc_vru_total', 'acc_vru_leves', 'acc_vru_graves',
            'acc_vru_mortales', 'acc_ponderado']
validacion_2025[cols_vru] = validacion_2025[cols_vru].fillna(0)

# Verificaciones
assert 'año_x' not in validacion_2025.columns, "⚠️ año duplicado — revisar merges"
assert validacion_2025['año'].eq(2025).all(),   "⚠️ año con valores inesperados"
assert (validacion_2025['acc_vru_leves'] + validacion_2025['acc_vru_graves'] +
        validacion_2025['acc_vru_mortales'] <= validacion_2025['acc_vru_total']).all(), \
    "⚠️ Inconsistencia VRU 2025"
print(f"✅ Columnas OK: {validacion_2025.columns.tolist()}")
print(f"✅ Distritos: {len(validacion_2025)} | Año único: {validacion_2025['año'].unique()}")

validacion_2025.to_csv(
    '/content/drive/MyDrive/TFM_Seguridad_Vial/outputs/validacion_2025.csv',
    index=False, encoding='utf-8-sig'
)
print('\n✅ validacion_2025.csv guardada')
print(validacion_2025[['distrito', 'año', 'acc_vru_total', 'acc_ponderado']].to_string())

✅ Columnas OK: ['distrito', 'año', 'acc_bici_total', 'acc_bici_leves', 'acc_bici_graves', 'acc_bici_mortales', 'acc_peatones_total', 'acc_peatones_leves', 'acc_peatones_graves', 'acc_peatones_mortales', 'acc_vru_total', 'acc_vru_leves', 'acc_vru_graves', 'acc_vru_mortales', 'acc_ponderado']
✅ Distritos: 21 | Año único: [2025]

✅ validacion_2025.csv guardada
               distrito   año  acc_vru_total  acc_ponderado
0            ARGANZUELA  2025            161            199
1               BARAJAS  2025             42             77
2           CARABANCHEL  2025            191            239
3                CENTRO  2025            248            245
4             CHAMARTÍN  2025            129            160
5              CHAMBERÍ  2025            140            202
6         CIUDAD LINEAL  2025            126            181
7   FUENCARRAL-EL PARDO  2025            111            200
8             HORTALEZA  2025             94            137
9                LATINA  2025           

**Celda 38 — Cerrar tabla maestra anual definitiva**

In [49]:
import shutil

outputs = '/content/drive/MyDrive/TFM_Seguridad_Vial/outputs'

shutil.copy(f'{outputs}/tabla_maestra_v5.csv',
            f'{outputs}/tabla_anual_final.csv')

df_final = pd.read_csv(f'{outputs}/tabla_anual_final.csv')
print('✅ tabla_anual_final.csv cerrada y lista para el índice descriptivo')
print(f'Filas: {len(df_final)} | Columnas: {len(df_final.columns)}')
print(f'Columnas: {df_final.columns.tolist()}')
print(f'Nulos: {df_final.isnull().sum().sum()}')

✅ tabla_anual_final.csv cerrada y lista para el índice descriptivo
Filas: 126 | Columnas: 26
Columnas: ['distrito', 'año', 'acc_bici_total', 'acc_bici_leves', 'acc_bici_graves', 'acc_bici_mortales', 'acc_peatones_total', 'acc_peatones_leves', 'acc_peatones_graves', 'acc_peatones_mortales', 'poblacion', 'n_paradas_emt', 'km_carril_bici', 'acc_vru_total', 'acc_vru_leves', 'acc_vru_graves', 'acc_vru_mortales', 'acc_ponderado', 'area_km2', 'pct_mayores_65', 'pct_menores_15', 'pct_red_30kmh', 'n_semaforos_peatonales', 'semaforos_km2', 'vel_media_vial', 'n_estaciones_bicimad']
Nulos: 0


**Celda 39 — Cerrar tabla mensual definitiva**

In [50]:
outputs = '/content/drive/MyDrive/TFM_Seguridad_Vial/outputs'

df_mensual = pd.read_csv(f'{outputs}/tabla_maestra_mensual_v3.csv')

# Añadimos estaciones BiciMAD (variable estática)
df_mensual = df_mensual.merge(n_estaciones_bicimad, on='distrito', how='left')
df_mensual['n_estaciones_bicimad'] = df_mensual['n_estaciones_bicimad'].fillna(0).astype(int)

# Guardar como tabla definitiva
df_mensual.to_csv(f'{outputs}/tabla_mensual_final.csv', index=False, encoding='utf-8-sig')

print('✅ tabla_mensual_final.csv cerrada y lista para modelo predictivo')
print(f'Filas: {len(df_mensual)} | Columnas: {len(df_mensual.columns)}')
print(f'Columnas: {df_mensual.columns.tolist()}')
print(f'Nulos: {df_mensual.isnull().sum().sum()}')
print(f'% ceros en acc_vru_graves: {(df_mensual["acc_vru_graves"]==0).mean():.1%}')

✅ tabla_mensual_final.csv cerrada y lista para modelo predictivo
Filas: 1512 | Columnas: 27
Columnas: ['distrito', 'año', 'mes', 'acc_bici_total', 'acc_bici_leves', 'acc_bici_graves', 'acc_bici_mortales', 'acc_peatones_total', 'acc_peatones_leves', 'acc_peatones_graves', 'acc_peatones_mortales', 'acc_vru_total', 'acc_vru_leves', 'acc_vru_graves', 'acc_vru_mortales', 'acc_ponderado', 'poblacion', 'n_paradas_emt', 'km_carril_bici', 'area_km2', 'pct_mayores_65', 'pct_menores_15', 'pct_red_30kmh', 'n_semaforos_peatonales', 'semaforos_km2', 'vel_media_vial', 'n_estaciones_bicimad']
Nulos: 0
% ceros en acc_vru_graves: 48.9%
